[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/whitney_forms/whitney_forms.ipynb)

# Whitney Forms

This example computes Whitney forms, the functions that spread values stored on the vertices and edges of a triangle mesh into fields across its triangles. A vertex value spreads as a hat, a ramp over the triangles around its vertex, so that neighbouring triangles meet without a seam. An edge value, the integral of a field along that edge, spreads as a field whose reading along the edge is the same from both of its triangles, even where the field itself jumps across it. A tetrahedron carries the same forms one dimension up, with face forms beside them that spread a flux through each face. Steady electric conduction through an anisotropic sheet puts these fields to work: the material acts on the field inside each triangle, and its response returns to the vertices and edges of the mesh.

A position is modelled as an antivector, so a triangle's signed area is the join of two of its sides, `(sides[0] & sides[1]) / 2`, an intrinsic patch. Dividing by twice that patch turns intrinsic directions into extrinsic forms, which read displacements by the regressive product without a metric; the corner gradients and the edge forms both arise this way. Assembled over the mesh, they become maps from vertex and edge values to fields on the faces, and these maps commute with the exterior derivative. The metric enters only with the material, a conductivity that maps the gradient of a potential to the current it drives; the gradient's adjugate carries that current back to the vertices, and the material pulled back through the gradient is the conductance between them.

In the notation of finite element exterior calculus, the hats and edge forms are the lowest-order Whitney forms, which span the spaces $\mathcal P_1^-\Lambda^0$ and $\mathcal P_1^-\Lambda^1$ of the discrete de Rham complex, and the Whitney maps $W^k$ from $k$-cochains to forms commute with the exterior derivative, $d\,W^k c = W^{k+1}\,d c$, where the $d$ on the right is the coboundary of the mesh.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import VGA2D as ga
from numga.sparse import spdiag
from examples import instantiate
from examples.surfaces.whitney_forms import render, scenarios
from examples.surfaces.whitney_forms.core import complements
from examples.util.animation import save_figure

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, mv = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                # Scalar+
Plane = ga.gatype.vector()                                 # Vector+
triangle = Mesh(3, [[0, 1, 2]])

np.set_printoptions(precision=4, suppress=True)

## Corner hats

Each corner hat rises from zero on the opposite edge to one at its corner. Around a shared vertex, the ramps meet continuously even though their slopes change from face to face. The gradient reads a displacement through `gradient & displacement`.

In [ ]:
n_triangle_subdivisions = 8
n_star_divisions = 2
disk_radius = 1.0

In [ ]:
corners, points = scenarios.single_triangle(n_triangle_subdivisions)  # [corners] Point-, [samples] Point-
star_mesh, star_vertices = scenarios.disk(disk_radius, n_star_divisions)  # Mesh, Point-[vertices]

# The signed patch and opposite edges determine the three affine hat gradients.
sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners.field())).batch()[:, 0]  # [corners] Point-
patch = (sides[0] & sides[1]) / 2                                      # [] Patch-
gradients = sides / (2 * patch)                                        # [corners] Plane+
hats = 1 + (gradients[:, None] & (points[None, :] - corners[:, None]))  # [corners, samples] Scalar+

# On the vertex star, assembling the three corner gradients per face maps vertex values to face gradients.
star_edges = star_mesh.d[0] * star_vertices                # Point-[edges]
star_sides = star_mesh.boundary[2, 1] * star_edges         # [corners] Point-[faces]
star_patches = (star_sides[0] & star_sides[1]) / 2         # Patch-[faces]
star_corner_gradients = star_sides / (2 * star_patches)    # [corners] Plane+[faces]
star_gradient = (spdiag(star_corner_gradients) * star_mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]
vertex_hat = as_scalar(np.arange(star_mesh.n_vertices) == 0).field()  # Scalar+[vertices]
star_gradients = star_gradient(vertex_hat)                 # Plane+[faces]

save_figure(
    render.draw_hat_0forms(corners, gradients, star_mesh, star_vertices, vertex_hat, star_gradients),
    "whitney_hat_0form",
);

The orange arrows show the constant gradient on each triangle; the blue shading shows the hat's height. The vertex star assembles those local ramps into one continuous hat.

In finite element notation, the corner hats are the barycentric functions $\lambda_i$, with constant differentials $d\lambda_i$ on each triangle.

In [ ]:
print("Triangle area:", patch.norm().to_array())

## Edge forms

An edge reading measures a field along a displacement. The corner gradient `sides / (2 * patch)` and the edge form `(points - corners) / (2 * patch)` share their division by twice the triangle's patch: the gradient divides the opposite edge, the form the displacement from the opposite corner. This makes an edge's own reading independent of the position along it, so adjacent triangles agree there.

In [ ]:
# The edge form reads the displacement from its opposite corner through the signed patch.
edge_forms = (points[None, :] - corners[:, None]) / (2 * patch)        # [corners, samples] Plane+
diamond_mesh, diamond_vertices, _ = scenarios.edge_diamond(n_triangle_subdivisions)
diamond_corners = diamond_mesh.incidence[2, 0] * diamond_vertices     # [corners] Point-[faces]
diamond_edges = diamond_mesh.d[0] * diamond_vertices                  # Point-[edges]
diamond_sides = diamond_mesh.boundary[2, 1] * diamond_edges           # [corners] Point-[faces]
diamond_patches = (diamond_sides[0] & diamond_sides[1]) / 2            # Patch-[faces]
# Unit circulation on the one interior edge, read on both faces through their edge forms,
# each signed by the sense in which its face runs along the edge.
shared_circulation = as_scalar(np.isin(np.arange(len(diamond_mesh.edges)), diamond_mesh.interior_edges)).field()  # Scalar+[edges]

shared = diamond_mesh.reading(diamond_mesh.interior_edges, 1)           # Scalar+[shared <- edges]
shared_edge, = (shared * diamond_edges).batch()                        # [] Point-
shared_midpoint, = (shared * (diamond_mesh.incidence[1, 0].mean(axis=0) * diamond_vertices)).batch()  # [] Point-
seam_forms = (shared_midpoint - diamond_corners) / (2 * diamond_patches)  # [corners] Plane+[faces]
seam_whitney1 = (spdiag(seam_forms) * diamond_mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]
left_at_seam, right_at_seam = seam_whitney1(shared_circulation).batch()  # [] Plane+, Plane+
# Corner values carry each affine form continuously over its triangle for drawing.
corner_forms = (corners[None, :] - corners[:, None]) / (2 * patch)    # [edges, corners] Plane+
corner_offsets = (diamond_corners[:, None] - diamond_corners[None, :]) / (2 * diamond_patches)
corner_whitney1 = (spdiag(corner_offsets) * diamond_mesh.boundary[2, 1]).sum(axis=1) * Scalar
shared_corner_forms = corner_whitney1(shared_circulation)            # [corners] Plane+[faces]
save_figure(
    render.draw_edge_1forms(corners, corner_forms, diamond_mesh, diamond_vertices, shared_corner_forms),
    "whitney_edge_1form",
);

The orange edge carries unit circulation. The streaks show the metric-dual directions of the forms, fading downstream with length proportional to magnitude. In the diamond they can turn across the shared edge while their tangential readings agree. The other two edges of each triangle carry zero tangential reading.

In the differential-form notation of finite element exterior calculus, the edge form is $w_{ij}=\lambda_i\,d\lambda_j-\lambda_j\,d\lambda_i$, with $dw_{ij}=2\,d\lambda_i\wedge d\lambda_j$.

In [ ]:
print("Normal reading (left, right):", (left_at_seam & mv.x.dual()).to_array(), (right_at_seam & mv.x.dual()).to_array())

## Forms on a tetrahedron

One dimension up, a tetrahedron in the algebra of space, VGA3D, carries forms of three degrees. Its signed volume, the join of three of its edges over six, is an intrinsic scalar and does the work of the triangle's patch. The gradient of a corner's hat is the patch of the face facing it, turned inward, over three times the volume, as the triangle's is the facing side over twice its signed area. The displacement from a corner over three times the volume, `(tet_points - tet_corners) / (3 * volume)`, is the 2-form of the face facing it, as `(points - corners) / (2 * patch)` is the triangle's edge form; it reads the flux through its own face and none through the others. Between the two sit the six edge forms: the patch the opposite edge sweeps to a point, over six times the volume, reads one along its own edge and zero along the others and circulates around the opposite edge, and its curl is that opposite edge over three times the volume. Curls and face forms alike are bivectors, planes of space. The helicity, the volume pairing of a field with its curl, pairs each edge form with each curl and links only opposite edges, the same way for every tetrahedron of the same sense.

In [ ]:
n_tet_subdivisions = 6

In [ ]:
tetrahedron, tet_vertices, tet_points = scenarios.single_tetrahedron(n_tet_subdivisions)  # Mesh, Point-[vertices], [samples] Point-[tets]
tet_corners = tetrahedron.incidence[3, 0] * tet_vertices                  # [corners] Point-[tets]
# The tetrahedron's edges in its own vertex order; the first three run from its first corner.
tet_edges = tetrahedron.oriented[3, 1] * (tetrahedron.d[0] * tet_vertices)  # [tet_edges] Point-[tets]
volume = (tet_edges[0] & tet_edges[1] & tet_edges[2]) / 6                 # Volume-[tets]
# Each face's patch, half the join of two of its sides, read as the face facing each corner, outward.
tet_sides = tetrahedron.boundary[2, 1] * (tetrahedron.d[0] * tet_vertices)  # [face_corners] Point-[faces]
face_patches = tetrahedron.boundary[3, 2] * ((tet_sides[0] & tet_sides[1]) / 2)  # [corners] Patch-[tets]
# The outward patch reads a step from a corner across to its face as three times the volume, and the hat falls
# by one along that step: its gradient is the patch turned inward, over three times the volume.
tet_gradients = -face_patches / (3 * volume)                              # [corners] Plane+[tets]

# The edge across each edge, run so that the two together run the tetrahedron's own sense.
opposite_tails, opposite_heads = complements(tetrahedron, 1).T
opposite_edges = tet_corners[opposite_heads] - tet_corners[opposite_tails]  # [tet_edges] Point-[tets]
# The patch the opposite edge sweeps to a point vanishes along that edge and, over six times the volume, reads
# one along the edge itself; its curl is the opposite edge over three times the volume, a plane of space.
tet_edge_forms = (opposite_edges[:, None] & (tet_points[None, :] - tet_corners[opposite_tails, None])) / (6 * volume)  # [tet_edges, samples] Plane+[tets]
tet_curls = opposite_edges / (3 * volume)                                 # [tet_edges] Bivector+[tets]
# Along the three faces at a corner the displacement from that corner lies in the face, while across the face
# facing it, it joins the face's outward patch to three times the volume.
tet_face_forms = (tet_points[None, :] - tet_corners[:, None]) / (3 * volume)  # [corners, samples] Bivector+[tets]

In [ ]:
# At the centroid every hat is a quarter, so each edge form is a quarter of the difference of its ends' gradients;
# a linear form against a constant curl integrates exactly there.
tails, heads = tetrahedron.corners[3, 1]
centroid_edge_forms = (tet_gradients[heads] - tet_gradients[tails]) / 4   # [tet_edges] Plane+[tets]
pair_densities = ((centroid_edge_forms[:, None] ^ tet_curls[None, :]) + (centroid_edge_forms[None, :] ^ tet_curls[:, None])) / 2  # [tet_edges, tet_edges] Trivector+[tets]
tet_helicity = pair_densities & volume                                    # [tet_edges, tet_edges] Scalar+[tets]

save_figure(
    render.draw_tet_whitney(tetrahedron, tet_vertices, tet_points, tet_edge_forms, tet_face_forms, volume),
    "whitney_tet_forms",
);

On the left, the arrows of the orange edge's form circulate around the blue edge across it; in the middle, the arrows of the form of the face shaded orange radiate from the corner it faces; on the right, rings around the blue edge, threaded by the orange edge, show the linkage that the helicity pairing measures.

In Whitney-form notation, the same edge and face fields read as $w_{ij}=\lambda_i\,d\lambda_j-\lambda_j\,d\lambda_i$ and $w_{ijk}=2(\lambda_i\,d\lambda_j\wedge d\lambda_k+\lambda_j\,d\lambda_k\wedge d\lambda_i+\lambda_k\,d\lambda_i\wedge d\lambda_j)$, and the helicity pairing as $\frac12\int_T(w_e\wedge dw_f+w_f\wedge dw_e)$, which is $\pm\frac16$ for opposite edges $e$ and $f$ and zero otherwise.

## Material response

Conductivity maps a potential gradient to a current direction. The gradient's adjugate brings the material's response back to the mesh, so changing the material leaves the interpolation intact. On edges, midpoint quadrature integrates the products of Whitney forms exactly, including on obtuse triangles.

In [ ]:
n_disk_divisions = 9
electrode_threshold = 0.8
fibre_angle = np.pi / 6
principal_conductivities = np.array([[1, 1], [10, 1]])     # [cases, axes]

Assembled over a whole mesh, the corner gradients and edge forms of its faces become maps from vertex and edge values to fields on the faces. They commute with the exterior derivative: the edge forms of a difference of vertex values are its gradient, `whitney1(d0(potential)) == gradient(potential)`, and their curl is the face form of the circulation, `curl(cochain) == whitney2(d1(cochain))`.

In [ ]:
mesh, vertices = scenarios.disk(disk_radius, n_disk_divisions)  # Mesh, Point-[vertices]
d0 = mesh.d[0] * Scalar                                    # (Scalar+ <- Scalar+)[edges <- vertices]
d1 = mesh.d[1] * Scalar                                    # (Scalar+ <- Scalar+)[faces <- edges]
edges = mesh.d[0] * vertices                               # Point-[edges]
face_corners = mesh.incidence[2, 0] * vertices             # [face_corners] Point-[faces]
face_sides = mesh.boundary[2, 1] * edges                   # [face_corners] Point-[faces]
patches = (face_sides[0] & face_sides[1]) / 2              # Patch-[faces]
areas = patches.norm()                                     # Scalar+[faces]
centres = face_corners.sum(axis=0) / 3                     # Point-[faces]

# Each corner hat's gradient is constant across its face; assembling by corner maps
# vertex potentials to piecewise-constant face gradients without any metric.
corner_gradients = face_sides / (2 * patches)              # [face_corners] Plane+[faces]
gradient = (spdiag(corner_gradients) * mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]

# At each face centroid the three barycentric hats equal 1/3, so the edge 1-form
# reduces to one third of the difference of its endpoints' corner gradients.
face_tails, face_heads = mesh.corners[2, 1]
centroid_forms = (corner_gradients[face_heads] - corner_gradients[face_tails]) / 3  # [face_corners] Plane+[faces]
whitney1 = (spdiag(centroid_forms) * mesh.oriented[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]

# The exterior derivative of each edge's Whitney 1-form is twice the wedge of its endpoints'
# gradients, matching the constant Whitney 2-form of unit integral over the face patch.
curls = 2 * (corner_gradients[face_tails] ^ corner_gradients[face_heads])  # [face_corners] Bivector+[faces]
curl = (spdiag(curls) * mesh.oriented[2, 1]).sum(axis=0) * Scalar      # (Bivector+ <- Scalar+)[faces <- edges]
whitney2 = spdiag(mv.xy / (mv.xy & patches) * Scalar)      # (Bivector+ <- Scalar+)[faces <- faces]

In [ ]:
fibre = (mv.xy * (-fibre_angle / 2)).exp() >> mv.x.dual()  # [] Point-
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                # [] Rotor+
directions = stack((fibre, quarter_turn >> fibre))         # [axes] Point-
conductivity = (directions * (Plane & directions) * principal_conductivities).sum(axis=-1)  # [cases] Point- <- Plane+

boundary = mesh.exterior(0)
abscissa = (mv.x & vertices).batch()                       # [vertices] Scalar+
electrodes = boundary[abscissa[boundary].norm() > electrode_threshold]
pinned = mesh.reading(electrodes)
prescribed = ~pinned * (pinned * (mv.x & vertices))        # Scalar+[vertices]
conductance = gradient.adjugate()(spdiag(areas * conductivity)(gradient))  # [cases] (Amount- <- Scalar+)[vertices <- vertices]
free_vertices = np.setdiff1d(np.arange(mesh.n_vertices), electrodes)
free = ~mesh.reading(free_vertices) * Scalar               # (Scalar+ <- Scalar+)[vertices <- free]
rhs = -free.adjugate()(conductance(prescribed))            # [cases] Amount-[free]
potential = prescribed + free(free.adjugate()(conductance(free)).solve(rhs))  # [cases] Scalar+[vertices]
current = -conductivity(gradient(potential))               # [cases] Point-[faces]

In [ ]:
save_figure(
    render.draw_anisotropic_flow(mesh, vertices, potential, current, fibre),
    "whitney_anisotropic_flow",
);

The left disk conducts equally in every direction. In the right disk, conductivity is greater along the tilted fibres, bending the current paths. Colour and contours show potential. The streaks fade downstream with length proportional to current magnitude on one scale shared by the two conductors; the faint chords mark the tilted fibres.

In the matrix notation of the finite element method, this pullback is the stiffness matrix $K_{pq}=\int_\Omega d\lambda_p\cdot(\sigma\,d\lambda_q)\,dA$; isotropic conductivity gives the cotangent Laplacian.

In [ ]:
print("Centre current (isotropic vs anisotropic):\n", (stack((mv.x, mv.y)) & current.batch()[:, 0, None]).to_array())

Two obtuse triangles sharing a long edge give that edge a negative weight in the circumcentric Hodge star of discrete exterior calculus; its Whitney conductance still stores positive energy.

In [ ]:
# Evaluating the linear Whitney 1-forms at the three edge midpoints integrates
# quadratic products over every triangle exactly.
midpoints = mesh.incidence[2, 1] * (mesh.incidence[1, 0].mean(axis=0) * vertices)  # [midpoints] Point-[faces]
midpoint_forms = (midpoints[:, None] - face_corners[None, :]) / (2 * patches)  # [midpoints, face_corners] Plane+[faces]
midpoint_whitney1 = (spdiag(midpoint_forms) * mesh.boundary[2, 1]).sum(axis=1) * Scalar  # [midpoints] (Plane+ <- Scalar+)[faces <- edges]
weights = spdiag(((areas / 3) * conductivity)[..., None])  # [cases, midpoints] (Point- <- Plane+)[faces <- faces]
edge_conductances = midpoint_whitney1.adjugate()(weights(midpoint_whitney1)).sum(axis=-1)  # [cases] (Amount- <- Scalar+)[edges <- edges]

obtuse_mesh, obtuse_vertices = scenarios.obtuse_diamond()
obtuse_corners = obtuse_mesh.incidence[2, 0] * obtuse_vertices  # [corners] Point-[faces]
obtuse_sides = obtuse_mesh.boundary[2, 1] * (obtuse_mesh.d[0] * obtuse_vertices)  # [corners] Point-[faces]
obtuse_patches = (obtuse_sides[0] & obtuse_sides[1]) / 2   # Patch-[faces]
obtuse_midpoints = obtuse_mesh.incidence[2, 1] * (obtuse_mesh.incidence[1, 0].mean(axis=0) * obtuse_vertices)  # [midpoints] Point-[faces]
obtuse_forms = (obtuse_midpoints[:, None] - obtuse_corners[None, :]) / (2 * obtuse_patches)  # [midpoints, corners] Plane+[faces]
obtuse_whitney1 = (spdiag(obtuse_forms) * obtuse_mesh.boundary[2, 1]).sum(axis=1) * Scalar  # [midpoints] (Plane+ <- Scalar+)[faces <- edges]
obtuse_weights = spdiag((obtuse_patches.norm() / 3) * Plane.dual())  # (Point- <- Plane+)[faces <- faces]
obtuse_edge_conductance = obtuse_whitney1.adjugate()(obtuse_weights(obtuse_whitney1)).sum(axis=0)  # (Amount- <- Scalar+)[edges <- edges]
shared_cochain = as_scalar(np.isin(np.arange(len(obtuse_mesh.edges)), obtuse_mesh.interior_edges)).field()  # Scalar+[edges]
obtuse_energy = -(obtuse_edge_conductance(shared_cochain) & shared_cochain).sites.sum()  # [] Scalar+

In [ ]:
# checks
corner_hats = 1 + (gradients[:, None] & (corners[None, :] - corners[:, None]))
np.testing.assert_allclose(corner_hats.kernel, np.eye(3)[..., None], atol=1e-14)
np.testing.assert_allclose(hats.sum(axis=0).kernel, 1.0, atol=1e-14)
np.testing.assert_allclose(gradients.sum(axis=0).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((1 + (gradients & (corners.mean() - corners))).kernel, 1 / 3, atol=1e-13)

first, second = complements(triangle, 0).T
alternating_forms = hats[first] * gradients[second, None] - hats[second] * gradients[first, None]
np.testing.assert_allclose((edge_forms - alternating_forms).kernel, 0.0, atol=1e-14)
midpoints = (corners[first] + corners[second]) / 2
midpoint_readings = (midpoints[None, :] - corners[:, None]) / (2 * patch) & sides[None, :]
np.testing.assert_allclose(midpoint_readings.kernel, np.eye(3)[..., None], atol=1e-14)

np.testing.assert_allclose((left_at_seam & shared_edge).kernel, 1.0, atol=1e-14)
np.testing.assert_allclose((right_at_seam & shared_edge).kernel, 1.0, atol=1e-14)

tet_hats = 1 + (tet_gradients[:, None] & (tet_points[None, :] - tet_corners[:, None]))
np.testing.assert_allclose(tet_hats.sum(axis=0).kernel, 1.0, atol=1e-13)
np.testing.assert_allclose(tet_gradients.sum(axis=0).kernel, 0.0, atol=1e-13)
np.testing.assert_allclose(face_patches.sum(axis=0).kernel, 0.0, atol=1e-13)
edge_midpoints = tetrahedron.incidence[3, 1] * (tetrahedron.incidence[1, 0].mean(axis=0) * tet_vertices)
midpoint_edge_forms = (opposite_edges[:, None] & (edge_midpoints[None, :] - tet_corners[opposite_tails, None])) / (6 * volume)
np.testing.assert_allclose((midpoint_edge_forms & tet_edges[None, :]).kernel, np.eye(6)[..., None, None], atol=1e-13)
face_centres = tetrahedron.incidence[3, 2] * (tetrahedron.incidence[2, 0].mean(axis=0) * tet_vertices)
centre_face_forms = (face_centres[None, :] - tet_corners[:, None]) / (3 * volume)
np.testing.assert_allclose((centre_face_forms & face_patches[None, :]).kernel, np.eye(4)[..., None, None], atol=1e-13)

alternating_edge_forms = tet_hats[tails] * tet_gradients[heads, None] - tet_hats[heads] * tet_gradients[tails, None]
np.testing.assert_allclose((tet_edge_forms - alternating_edge_forms).kernel, 0.0, atol=1e-13)
# The corners of the face facing each corner, wound outward.
face_first, face_second, face_third = complements(tetrahedron, 0).T
alternating_face_forms = 2 * (tet_hats[face_first] * (tet_gradients[face_second, None] ^ tet_gradients[face_third, None])
                              + tet_hats[face_second] * (tet_gradients[face_third, None] ^ tet_gradients[face_first, None])
                              + tet_hats[face_third] * (tet_gradients[face_first, None] ^ tet_gradients[face_second, None]))
np.testing.assert_allclose((tet_face_forms - alternating_face_forms).kernel, 0.0, atol=1e-13)

# A sheared, stretched tetrahedron of the same sense pairs its edge forms and curls the same way.
deformed_vertices = scenarios.deformed_tetrahedron()
deformed_edges = tetrahedron.oriented[3, 1] * (tetrahedron.d[0] * deformed_vertices)
deformed_volume = (deformed_edges[0] & deformed_edges[1] & deformed_edges[2]) / 6
deformed_sides = tetrahedron.boundary[2, 1] * (tetrahedron.d[0] * deformed_vertices)
deformed_gradients = -(tetrahedron.boundary[3, 2] * ((deformed_sides[0] & deformed_sides[1]) / 2)) / (3 * deformed_volume)
deformed_forms = (deformed_gradients[heads] - deformed_gradients[tails]) / 4
deformed_curls = 2 * (deformed_gradients[tails] ^ deformed_gradients[heads])
deformed_densities = ((deformed_forms[:, None] ^ deformed_curls[None, :]) + (deformed_forms[None, :] ^ deformed_curls[:, None])) / 2
deformed_helicity = deformed_densities & deformed_volume
np.testing.assert_allclose((tet_helicity - deformed_helicity).kernel, 0.0, atol=1e-13)
# Two edges link where they share no corner, with the sense of the tetrahedron's corners in the order they list
# them, over six.
edge_corners = tetrahedron.corners[3, 1].T
edge_pairs = np.concatenate(np.broadcast_arrays(edge_corners[:, None], edge_corners[None, :]), axis=-1)
disjoint = (edge_corners[:, None, :, None] != edge_corners[None, :, None, :]).all(axis=(-1, -2))
linking = np.where(disjoint, mesh_module.simplex_parity(edge_pairs), 0) / 6
np.testing.assert_allclose(tet_helicity.kernel[..., 0, 0], linking, atol=1e-13)

linear_potential = (mv.x & vertices) * 0.7 - (mv.y & vertices) * 1.1
np.testing.assert_allclose(d1(d0(linear_potential)).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((whitney1(d0(linear_potential)) - gradient(linear_potential)).kernel, 0.0, atol=1e-14)
test_cochain = d0(linear_potential) + as_scalar(np.sin(np.arange(len(mesh.edges)))).field()
np.testing.assert_allclose((curl(test_cochain) - whitney2(d1(test_cochain))).kernel, 0.0, atol=1e-12)

quadratic_potential = (mv.x & vertices).squared() - 0.5 * (mv.y & vertices)
np.testing.assert_allclose(
    (d0.adjugate()(edge_conductances(d0(quadratic_potential))) - conductance(quadratic_potential)).kernel,
    0.0, atol=1e-13,
)
np.testing.assert_allclose((conductance - conductance.adjugate()).cells.kernel, 0.0, atol=1e-13)
assert obtuse_energy > 0

np.testing.assert_allclose(free.adjugate()(conductance(potential)).kernel, 0.0, atol=1e-11)